# Model Comparison - Train LSTM/GRU
Notebook นี้ฝึกโมเดล **LSTM + GRU** จาก sequence (window) เพื่อใช้เปรียบเทียบกับโมเดลอื่นใน Results.

In [ ]:
import os
import sys
import json
import time
import random
import pickle
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sys.path.insert(0, os.path.abspath('..'))
import src.config as config
from src.data.processing import transform_dataset, create_sequences, video_based_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [2]:
WINDOW_SIZE = 10
STEP_SIZE = 1
BATCH_SIZE = 64
EPOCHS = 25
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 128
PATIENCE = 6

data_dir = os.path.join(config.DATASET_DIR, 'by_class')
dfs = []
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith('.csv'):
        continue
    csv_path = os.path.join(data_dir, file_name)
    df_raw = pd.read_csv(csv_path).dropna()
    df_pro = transform_dataset(df_raw)
    if not df_pro.empty:
        dfs.append(df_pro)
        print(f'Loaded {file_name}: {len(df_pro)}')

if not dfs:
    raise RuntimeError('No data loaded from dataset/by_class')

df_all = pd.concat(dfs, ignore_index=True)
X_seq, y_seq = create_sequences(df_all, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
print('X_seq shape:', X_seq.shape)
print('y_seq shape:', y_seq.shape)

Loaded block.csv: 500
Loaded dodge_back.csv: 300
Loaded dodge_front.csv: 300
Loaded dodge_left.csv: 300
Loaded dodge_right.csv: 300
Loaded final_skill.csv: 300
Loaded left_punch.csv: 300
Loaded neutral.csv: 500
Loaded right_punch.csv: 300
X_seq shape: (3019, 10, 108)
y_seq shape: (3019,)


In [ ]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_seq)

# Use VIDEO-BASED SPLIT to prevent data leakage
# chunk_size=50 assumes ~50 consecutive sequences per video chunk
print('\n⚠️  Using VIDEO-BASED SPLIT (no data leakage)')
X_train, X_val, X_test, y_train, y_val, y_test = video_based_split(
    X_seq, y_encoded, test_size=0.2, val_size=0.15, chunk_size=50, random_state=SEED
)

scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

def scale_sequences(x):
    shape = x.shape
    x_scaled = scaler.transform(x.reshape(-1, shape[-1])).reshape(shape)
    return x_scaled.astype(np.float32)

X_train = scale_sequences(X_train)
X_val = scale_sequences(X_val)
X_test = scale_sequences(X_test)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=False)

print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))
print('✓ Honest train/val/test split completed')

Train: 2052 Val: 363 Test: 604


In [ ]:
class LSTMGRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.gru = nn.GRU(
            input_size=hidden_dim * 2,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.gru(x)
        x = x[:, -1, :]
        return self.classifier(x)

model = LSTMGRUClassifier(input_dim=X_train.shape[-1], hidden_dim=HIDDEN_SIZE, num_classes=len(encoder.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_state = None
best_val_loss = float('inf')
wait = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_true = []
    val_pred = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            pred = logits.argmax(dim=1)
            val_true.extend(yb.cpu().numpy())
            val_pred.extend(pred.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_acc = accuracy_score(val_true, val_pred)
    print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping triggered')
            break

if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
model.eval()
all_true = []
all_pred = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.extend(pred)
        all_true.extend(yb.numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)

acc = accuracy_score(all_true, all_pred)
macro_f1 = f1_score(all_true, all_pred, average='macro')

y_true_txt = encoder.inverse_transform(all_true)
y_pred_txt = encoder.inverse_transform(all_pred)

print(f'Test Accuracy: {acc:.4f}')
print(f'Macro F1: {macro_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_true_txt, y_pred_txt))

cm = confusion_matrix(y_true_txt, y_pred_txt, labels=encoder.classes_)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title(f'LSTM/GRU Confusion Matrix (Acc: {acc*100:.2f}%)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
sample = torch.tensor(X_test[:1], dtype=torch.float32).to(device)
with torch.no_grad():
    for _ in range(50):
        _ = model(sample)

runs = 300
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(runs):
        _ = model(sample)
t1 = time.perf_counter()

latency_ms = (t1 - t0) / runs * 1000
print(f'Inference latency (1 sequence): {latency_ms:.4f} ms')

In [ ]:
# Create timestamped folder for this run
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_folder = f'lstm_gru_{timestamp}'

models_out = os.path.join(config.MODELS_DIR, 'model_comparison', 'lstm_gru', run_folder)
reports_out = os.path.join(config.PROJECT_ROOT, 'reports', 'model_comparison', 'lstm_gru', run_folder)
os.makedirs(models_out, exist_ok=True)
os.makedirs(reports_out, exist_ok=True)

print(f'Saving to run folder: {run_folder}')

# Save model artifacts
torch.save({
    'state_dict': model.state_dict(),
    'input_dim': int(X_train.shape[-1]),
    'hidden_dim': int(HIDDEN_SIZE),
    'num_classes': int(len(encoder.classes_)),
    'window_size': int(WINDOW_SIZE),
    'classes': list(encoder.classes_)
}, os.path.join(models_out, 'lstm_gru_model.pt'))

with open(os.path.join(models_out, 'lstm_gru_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(models_out, 'lstm_gru_label_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)

# Save confusion matrix
cm_path = os.path.join(reports_out, 'confusion_matrix.png')
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title(f'LSTM/GRU Confusion Matrix (Acc: {acc*100:.2f}%)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(cm_path, dpi=160)
plt.close()

# Save metrics
metrics = {
    'model': 'LSTM/GRU',
    'timestamp': timestamp,
    'run_folder': run_folder,
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'inference_latency_ms': float(latency_ms),
    'train_samples': int(len(X_train)),
    'val_samples': int(len(X_val)),
    'test_samples': int(len(X_test)),
    'window_size': int(WINDOW_SIZE),
    'num_features': int(X_train.shape[-1]),
    'hidden_size': int(HIDDEN_SIZE),
    'classes': list(encoder.classes_)
}

metrics_path = os.path.join(reports_out, 'metrics.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# Also save to root comparison folder for summary
root_metrics_path = os.path.join(config.PROJECT_ROOT, 'reports', 'model_comparison', 'lstm_gru_metrics.json')
with open(root_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f'✓ Saved model to: {models_out}')
print(f'✓ Saved reports to: {reports_out}')
print(f'✓ Metrics: {metrics_path}')